# Notebook 12 – Feature Leakage

This notebook covers what feature leakage is, the different ways it sneaks into a model, and side-by-side examples showing incorrect feature engineering that causes leakage versus correct feature engineering that avoids it.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['High_Value'] = (df['TotalPrice'] > df['TotalPrice'].median()).astype(int)
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,High_Value
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,1
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,1
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,1
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,1
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,1


**Code Explanation:** The dataset is loaded and cleaned, TotalPrice is calculated, InvoiceDate is converted into a real date type, and a High_Value target column is created based on whether TotalPrice is above the median, which is used to demonstrate leakage in the examples below.

## 1. What is Feature Leakage?

**Definition:** Feature leakage happens when information that would not be available at prediction time accidentally gets used to train the model, making it perform unrealistically well during testing but poorly in the real world.

**Example:** Using a feature that is calculated directly from the target itself, so the model is basically cheating by seeing the answer in disguise.

**Why it is used?** Understanding leakage matters because a leaky model can look excellent in testing and then fail completely once it faces new, real data.

In [2]:
df['Leaky_Feature'] = df['TotalPrice'] / df['TotalPrice'].mean()
df[['TotalPrice', 'High_Value', 'Leaky_Feature']].head()

,TotalPrice,High_Value,Leaky_Feature
0,15.30,1,0.749932
1,20.34,1,0.996968
2,22.00,1,1.078333
3,20.34,1,0.996968
4,20.34,1,0.996968


**Code Explanation:** Leaky_Feature is built directly from TotalPrice, which is the same value the High_Value target is based on, so this feature secretly carries the answer inside it.

## Incorrect Feature Engineering to Correct Feature Engineering (Target Leakage)

**Incorrect:** Including Leaky_Feature in the model, since it is mathematically tied to the target.

**Correct:** Only using features that would genuinely be known before TotalPrice is calculated, like Quantity and UnitPrice on their own, without any feature built from the target.

In [3]:
X_leaky = df[['Quantity', 'UnitPrice', 'Leaky_Feature']].sample(n=5000, random_state=1)
y_leaky = df.loc[X_leaky.index, 'High_Value']
X_train, X_test, y_train, y_test = train_test_split(X_leaky, y_leaky, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
preds = model.predict(X_test)
accuracy_score(y_test, preds)

0.996

In [4]:
X_clean = df[['Quantity', 'UnitPrice']].sample(n=5000, random_state=1)
y_clean = df.loc[X_clean.index, 'High_Value']
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_clean, y_clean, test_size=0.3, random_state=42)
model2 = LogisticRegression(max_iter=1000)
model2.fit(X_train2, y_train2)
preds2 = model2.predict(X_test2)
accuracy_score(y_test2, preds2)

0.816

**Code Explanation:** The leaky model scores an unrealistic 99.6 percent accuracy because Leaky_Feature basically reveals the target, while the clean model without that feature scores a much more believable 81.6 percent, showing what genuine, honest performance looks like.

## 2. Target Leakage

**Definition:** A specific type of leakage where a feature contains information derived from or highly dependent on the target variable.

**Example:** Leaky_Feature above is a textbook case, since it is a direct transformation of TotalPrice, the same value the target is based on.

**Why it is used?** Recognizing target leakage helps catch the most damaging and easy-to-miss kind of leakage before it reaches a production model.

## 3. Train-Test Leakage

**Definition:** Leakage that happens when information from the test set accidentally influences how the training data is prepared, like scaling using statistics from the full dataset instead of just the training set.

**Example:** Fitting a StandardScaler on the entire dataset before splitting it into train and test.

**Why it is used?** Prevents the model from getting an unfair sneak peek at the test data's distribution during preprocessing.

In [5]:
feats = df[['Quantity', 'UnitPrice']].sample(n=5000, random_state=2)
target = df.loc[feats.index, 'High_Value']
scaler_wrong = StandardScaler()
scaled_all = scaler_wrong.fit_transform(feats)
Xtr_w, Xte_w, ytr_w, yte_w = train_test_split(scaled_all, target, test_size=0.3, random_state=42)
scaler_wrong.mean_

array([12.1926  ,  3.666936])

## Incorrect Feature Engineering to Correct Feature Engineering (Train-Test Leakage)

**Incorrect:** Fitting the scaler on the full dataset before splitting, so the test set influences the mean and standard deviation used for scaling.

**Correct:** Splitting the data first, then fitting the scaler only on the training set, and applying that same fitted scaler to the test set.

In [6]:
Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(feats, target, test_size=0.3, random_state=42)
scaler_correct = StandardScaler()
scaler_correct.fit(Xtr_c)
scaler_correct.mean_

array([12.60857143,  3.88815429])

**Code Explanation:** The wrong version calculates the mean using all 5000 rows including the test set, giving [12.19, 3.67]. The correct version calculates the mean using only the training rows, giving a slightly different [12.61, 3.89], which is the honest, leakage-free way to scale data.

## 4. Temporal Leakage

**Definition:** Leakage that happens when future information is used to predict the past, which is a problem specifically for time-based data.

**Example:** Randomly shuffling transactions before splitting into train and test, instead of keeping the split in proper time order.

**Why it is used?** Prevents a model from learning from the future when it should only ever learn from the past, which is exactly how it would work in real deployment.

In [7]:
df_sorted = df.sort_values('InvoiceDate')
cutoff = df_sorted['InvoiceDate'].quantile(0.8)
train_time = df_sorted[df_sorted['InvoiceDate'] <= cutoff]
test_time = df_sorted[df_sorted['InvoiceDate'] > cutoff]
print(cutoff)
print(len(train_time), len(test_time))

2011-11-02 10:36:00
325479 81350


## Incorrect Feature Engineering to Correct Feature Engineering (Temporal Leakage)

**Incorrect:** Using train_test_split with random shuffling on time-based data, which can put later transactions in the training set and earlier ones in the test set.

**Correct:** Sorting by InvoiceDate first, then placing the earliest 80 percent of transactions in the training set and the most recent 20 percent in the test set, matching how the model would actually be used going forward.

**Code Explanation:** InvoiceDate is sorted, a cutoff date is chosen at the 80th percentile, and everything before that date becomes the training set while everything after becomes the test set, keeping time order intact instead of mixing past and future.

## 5. Post-Outcome Features

**Definition:** Features that are only known after the outcome has already happened, so they should never be available at prediction time.

**Example:** Using a Refund_Issued column to predict whether an order will be cancelled, when refunds only happen after a cancellation is already known.

**Why it is used?** Helps catch features that sound useful but are actually only recorded after the event the model is trying to predict.

In [8]:
df['Is_Cancelled'] = df['InvoiceNo'].astype(str).str.startswith('C')
df[['InvoiceNo', 'Is_Cancelled']].head()

,InvoiceNo,Is_Cancelled
0,536365,False
1,536365,False
2,536365,False
3,536365,False
4,536365,False


## Incorrect Feature Engineering to Correct Feature Engineering (Post-Outcome Features)

**Incorrect:** Using Is_Cancelled as an input feature to predict TotalPrice or High_Value, since cancellation is only known after the transaction is already recorded.

**Correct:** Excluding Is_Cancelled from the feature set entirely when it represents something that happens after or alongside the outcome being predicted, and only using features genuinely available beforehand, like Quantity and UnitPrice.

**Code Explanation:** Is_Cancelled is derived from InvoiceNo starting with the letter C, which marks a cancelled order, and since this status is only known after the order outcome is decided, it should never be fed into a model as an input feature.

## 6. Leakage Through Aggregations

**Definition:** Leakage that happens when an aggregated feature, like a customer average, is calculated using the entire dataset including rows that should be in the test set.

**Example:** Calculating each customer's average spend using all their transactions, even ones that later end up in the test set.

**Why it is used?** Aggregation leakage is easy to miss because the feature looks harmless, but it quietly leaks test information into training.

In [9]:
cust_avg_all = df.groupby('CustomerID')['TotalPrice'].transform('mean')
cust_avg_all.head()

0    16.950737
1    16.950737
2    16.950737
3    16.950737
4    16.950737
Name: TotalPrice, dtype: float64

## Incorrect Feature Engineering to Correct Feature Engineering (Aggregation Leakage)

**Incorrect:** Using transform to compute each customer's average spend across the whole dataset before splitting, so test rows contribute to a value used to predict those same test rows.

**Correct:** Computing the customer average only from the training set, then mapping those training averages onto the test set, so a customer's test rows never influence their own average.

In [10]:
train_df, test_df = train_test_split(df.sample(n=5000, random_state=3), test_size=0.3, random_state=42)
cust_avg_train = train_df.groupby('CustomerID')['TotalPrice'].mean()
test_df = test_df.copy()
test_df['Cust_Avg_Correct'] = test_df['CustomerID'].map(cust_avg_train)
test_df[['CustomerID', 'Cust_Avg_Correct']].head()

,CustomerID,Cust_Avg_Correct
425132,17837.0,12.7375
204674,16448.0,NaN
81134,15719.0,0.7000
58716,16059.0,NaN
311679,13728.0,NaN


**Code Explanation:** The average is calculated only from train_df, then mapped onto test_df by CustomerID. Some test customers were never seen in training, which correctly shows up as missing values instead of a fake number, proving no test information leaked into the calculation.

## 7. Leakage Through Target Encoding

**Definition:** Leakage that happens when a categorical feature is replaced with a statistic of the target, like the average target value per category, calculated using the full dataset instead of just the training data.

**Example:** Replacing Country with the average High_Value rate for that country, calculated using all rows including future test rows.

**Why it is used?** Target encoding is powerful but extremely leakage-prone if not done carefully, since it directly bakes target information into a feature.

In [11]:
country_target_all = df.groupby('Country')['High_Value'].transform('mean')
country_target_all.head()

0    0.466008
1    0.466008
2    0.466008
3    0.466008
4    0.466008
Name: High_Value, dtype: float64

## Incorrect Feature Engineering to Correct Feature Engineering (Target Encoding Leakage)

**Incorrect:** Encoding Country using the average High_Value across the entire dataset, which means the target itself was used to build a feature before the train-test split even happened.

**Correct:** Splitting the data first, computing the average High_Value per country using only the training set, then mapping those training averages onto the test set, optionally smoothing rare categories to avoid overfitting to small sample sizes.

**Code Explanation:** transform calculates the mean target value per country using the whole dataset at once, which means test rows for a given country directly influence the encoded value used for that same country in training, a classic and common target encoding leak.

## 8. Detecting Leakage

**Definition:** The process of checking a model and its features for signs that leakage might be happening.

**Example:** A model scoring 99.6 percent accuracy on a real-world business problem is a major red flag, since real data is rarely that clean or predictable.

**Why it is used?** Catching leakage early avoids deploying a model that looks great in testing but fails immediately in the real world.

In [12]:
print("Leaky model accuracy:", accuracy_score(y_test, preds))
print("Clean model accuracy:", accuracy_score(y_test2, preds2))

Leaky model accuracy: 0.996
Clean model accuracy: 0.816


**Code Explanation:** Comparing the two accuracy scores side by side highlights exactly the kind of suspicious gap that should trigger a closer look for leakage, since a jump from 81.6 percent to 99.6 percent from adding one feature is rarely a genuine improvement.

## 9. Preventing Leakage

**Prevention checklist**
- Never include a feature that is mathematically derived from the target
- Always split data into train and test before fitting any scaler, encoder, or aggregation
- For time-based data, split by date instead of shuffling randomly
- Double check any feature that seems to only be known after the outcome happens
- Compute aggregated features like customer averages and target encodings using only the training set, then map them onto the test set
- Be suspicious of unusually high accuracy or performance, and investigate before trusting it